In [6]:
# ============================================================================
# OPTIMIZED MOE SYSTEM WITH BATCH PROCESSING
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
import re
import time

# ============================================================================
# STEP 1: Define URLFeatures Class
# ============================================================================

class URLFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, urls):
        urls = np.array(urls).reshape(-1)
        feats = np.array([
            [
                len(u),
                u.count('-'),
                u.count('@'),
                u.count('?'),
                u.count('='),
                u.count('.'),
                int(u.startswith("https")),
                int(u.count("//") > 1)
            ]
            for u in urls
        ])
        return csr_matrix(feats)

# ============================================================================
# STEP 2: Define GatingNetwork Class
# ============================================================================

class GatingNetwork(nn.Module):
    def __init__(self, input_size=8, hidden_size=64, num_experts=2):
        super(GatingNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_experts)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        weights = self.softmax(x)
        return weights

# ============================================================================
# STEP 3: Load Expert Models
# ============================================================================

print("Loading Expert Models...")

URL_MODEL_PATH = r"C:\Users\angelo\Downloads\THESIS\URL_Expert-20251210T060216Z-1-001\URL_Expert\Notebook and Model\url_expert_1.pkl"
expert_1 = joblib.load(URL_MODEL_PATH)
print("✓ Expert 1 (URL) loaded")

TEXT_MODEL_PATH = r"C:\Users\angelo\Downloads\THESIS\distilbert_phishing_model"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
expert_2 = AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL_PATH)

# OPTIMIZATION: Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
expert_2.to(device)
expert_2.eval()
print(f"✓ Expert 2 (Text) loaded on {device}")

# ============================================================================
# STEP 4: Load Trained Gating Network
# ============================================================================

print("Loading Trained Gating Network...")
gating_net = GatingNetwork(input_size=8, hidden_size=64, num_experts=2)
gating_net.load_state_dict(torch.load('gating_network.pth'))
gating_net.to(device)
gating_net.eval()
print("✓ Gating Network loaded")

print("\n Complete MoE system loaded!\n")

# ============================================================================
# STEP 5: Phrase Dictionary and Helper Functions
# ============================================================================

phrase_dict = {
    'urgent': 0.3,
    'verify account': 0.5,
    'suspended': 0.4,
    'click here': 0.3,
    'confirm your': 0.4,
    'congratulations': 0.3,
    'winner': 0.4,
    'limited time': 0.3,
    'act now': 0.3,
    'security alert': 0.5,
    'claim': 0.3,
    'prize': 0.3,
    'free': 0.2,
    'bonus': 0.2,
}

def preprocess_text(text):
    if pd.isna(text) or text == "":
        return ""
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def calculate_phrase_score(text, phrase_dict):
    if not text:
        return 0.0
    text_lower = text.lower()
    score = 0.0
    for phrase, weight in phrase_dict.items():
        if phrase in text_lower:
            score += weight
    return min(score, 1.0)

def extract_gating_features(text, url, phrase_score):
    url_present = 1 if (url and not pd.isna(url) and url != "") else 0
    message_length = len(text.split()) if text else 0
    emoji_count = len(re.findall(r'[^\w\s,]', text)) if text else 0
    hashtag_count = text.count('#') if text else 0
    url_count = len(re.findall(r'http\S+', text)) if text else 0
    
    if text and len(text) > 0:
        capital_ratio = sum(1 for c in text if c.isupper()) / len(text)
    else:
        capital_ratio = 0.0
    
    embedding_summary = 0.0
    
    features = np.array([
        url_present,
        phrase_score,
        message_length,
        emoji_count,
        hashtag_count,
        url_count,
        capital_ratio,
        embedding_summary
    ], dtype=np.float32)
    
    return features

# ============================================================================
# STEP 6: OPTIMIZED Batch Prediction Function
# ============================================================================

def predict_batch(texts, urls, batch_size=64, show_progress=False):
    """
    OPTIMIZED batch prediction - 3-10x faster than single predictions!
    
    Args:
        texts: List of text messages
        urls: List of URLs
        batch_size: Number of samples to process together (default 64)
        show_progress: Show progress during processing
    
    Returns:
        Dictionary with predictions, confidences, weights, and timing info
    """
    
    num_samples = len(texts)
    
    results = {
        'predictions': [],
        'confidences': [],
        'url_weights': [],
        'text_weights': [],
        'url_predictions': [],
        'text_predictions': [],
        'timings': {
            'preprocessing': [],
            'url_expert': [],
            'text_expert': [],
            'gating_network': [],
            'combination': [],
            'total': []
        }
    }
    
    start_overall = time.perf_counter()
    
    for batch_idx in range(0, num_samples, batch_size):
        batch_end = min(batch_idx + batch_size, num_samples)
        batch_texts = texts[batch_idx:batch_end]
        batch_urls = urls[batch_idx:batch_end]
        batch_len = len(batch_texts)
        
        start_batch = time.perf_counter()
        
        # ===== PREPROCESSING (Vectorized) =====
        start = time.perf_counter()
        processed_texts = [preprocess_text(t) for t in batch_texts]
        phrase_scores = [calculate_phrase_score(t, phrase_dict) for t in processed_texts]
        preprocess_time = time.perf_counter() - start
        
        # ===== URL EXPERT (Batch) =====
        start = time.perf_counter()
        valid_urls = [str(u) if u and str(u).strip() else "" for u in batch_urls]
        
        if any(valid_urls):
            try:
                url_df = pd.DataFrame({'url': valid_urls})
                url_probs_batch = expert_1.predict_proba(url_df)
            except:
                url_probs_batch = np.array([[0.5, 0.5]] * batch_len)
        else:
            url_probs_batch = np.array([[0.5, 0.5]] * batch_len)
        url_time = time.perf_counter() - start
        
        # ===== TEXT EXPERT (Batch) - KEY OPTIMIZATION =====
        start = time.perf_counter()
        valid_texts = [t if t else "" for t in processed_texts]
        
        if any(valid_texts):
            try:
                # OPTIMIZATION: Batch tokenization with fixed padding
                inputs = tokenizer(
                    valid_texts,
                    return_tensors='pt',
                    padding='max_length',  # Fixed padding for consistent speed
                    truncation=True,
                    max_length=128
                )
                
                # OPTIMIZATION: Move inputs to same device as model
                inputs = {k: v.to(device) for k, v in inputs.items()}
                
                with torch.no_grad():
                    outputs = expert_2(**inputs)
                    text_probs_batch = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            except Exception as e:
                text_probs_batch = np.array([[0.5, 0.5]] * batch_len)
        else:
            text_probs_batch = np.array([[0.5, 0.5]] * batch_len)
        text_time = time.perf_counter() - start
        
        # ===== GATING NETWORK (Batch) =====
        start = time.perf_counter()
        gating_features_batch = np.array([
            extract_gating_features(t, u, ps)
            for t, u, ps in zip(processed_texts, batch_urls, phrase_scores)
        ])
        gating_input = torch.FloatTensor(gating_features_batch).to(device)
        
        with torch.no_grad():
            expert_weights_batch = gating_net(gating_input).cpu().numpy()
        gating_time = time.perf_counter() - start
        
        # ===== COMBINE PREDICTIONS (Vectorized) =====
        start = time.perf_counter()
        final_probs_batch = (
            expert_weights_batch[:, 0:1] * url_probs_batch +
            expert_weights_batch[:, 1:2] * text_probs_batch
        )
        
        predictions_batch = ["PHISHING ⚠" if p[1] > 0.5 else "SAFE " for p in final_probs_batch]
        confidences_batch = [max(p) * 100 for p in final_probs_batch]
        url_predictions_batch = ['PHISHING' if p[1] > 0.5 else 'SAFE' for p in url_probs_batch]
        text_predictions_batch = ['PHISHING' if p[1] > 0.5 else 'SAFE' for p in text_probs_batch]
        combination_time = time.perf_counter() - start
        
        batch_total_time = time.perf_counter() - start_batch
        
        # Store results
        results['predictions'].extend(predictions_batch)
        results['confidences'].extend(confidences_batch)
        results['url_weights'].extend(expert_weights_batch[:, 0] * 100)
        results['text_weights'].extend(expert_weights_batch[:, 1] * 100)
        results['url_predictions'].extend(url_predictions_batch)
        results['text_predictions'].extend(text_predictions_batch)
        
        # Store timing (distributed across batch)
        results['timings']['preprocessing'].extend([preprocess_time / batch_len] * batch_len)
        results['timings']['url_expert'].extend([url_time / batch_len] * batch_len)
        results['timings']['text_expert'].extend([text_time / batch_len] * batch_len)
        results['timings']['gating_network'].extend([gating_time / batch_len] * batch_len)
        results['timings']['combination'].extend([combination_time / batch_len] * batch_len)
        results['timings']['total'].extend([batch_total_time / batch_len] * batch_len)
        
        if show_progress and (batch_end % 1000 == 0 or batch_end == num_samples):
            elapsed = time.perf_counter() - start_overall
            rate = batch_end / elapsed
            print(f"  Processed {batch_end:,}/{num_samples:,} | Rate: {rate:.1f} rec/sec")
    
    return results

# ============================================================================
# STEP 7: Single Prediction (for testing individual samples)
# ============================================================================

def predict_single(text, url):
    """Single prediction using batch function (for consistency)"""
    results = predict_batch([text], [url], batch_size=1)
    
    return {
        'prediction': results['predictions'][0],
        'confidence': results['confidences'][0],
        'url_weight': results['url_weights'][0],
        'text_weight': results['text_weights'][0],
        'url_prediction': results['url_predictions'][0],
        'text_prediction': results['text_predictions'][0],
    }

def test_sample(input_text):
    """Auto-detect URL and text, then predict"""
    
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    urls = re.findall(url_pattern, input_text)
    
    if urls:
        url = urls[0]
        text = re.sub(url_pattern, '', input_text).strip()
    else:
        url = ""
        text = input_text.strip()
    
    results = predict_single(text, url)
    
    print("=" * 70)
    print(" PREDICTION RESULTS (Optimized MoE System)")
    print("=" * 70)
    if text:
        print(f" Text: {text[:80]}..." if len(text) > 80 else f"📝 Text: {text}")
    if url:
        print(f"🔗 URL: {url}")
    print("\n" + "-" * 70)
    print(f" Learned Expert Weights:")
    print(f"   URL Expert:  {results['url_weight']:.1f}%")
    print(f"   Text Expert: {results['text_weight']:.1f}%")
    print("\n Individual Expert Predictions:")
    print(f"   URL Expert:  {results['url_prediction']}")
    print(f"   Text Expert: {results['text_prediction']}")
    print("-" * 70)
    print(f" FINAL PREDICTION: {results['prediction']}")
    print(f" Confidence: {results['confidence']:.2f}%")
    print("=" * 70)
    print()
    
    return results

# ============================================================================
# STEP 8: Large Dataset Testing with Optimization
# ============================================================================

def test_large_dataset(text_dataset_path, url_dataset_path, num_records=10000, 
                      text_col='TEXT', url_col='URL', batch_size=64):
    """
    Test on large datasets with OPTIMIZED batch processing
    
    Args:
        text_dataset_path: Path to text dataset CSV
        url_dataset_path: Path to URL dataset CSV
        num_records: Number of records from EACH dataset
        text_col: Text column name
        url_col: URL column name
        batch_size: Batch size (32, 64, 128, or 256)
    """
    
    print("\n" + "=" * 70)
    print(f"OPTIMIZED BATCH PROCESSING (batch_size={batch_size})".center(70))
    print(f"Device: {device}".center(70))
    print("=" * 70)
    
    # Load datasets with encoding handling
    print(f"  Loading text dataset from: {text_dataset_path}")
    try:
        df_text = pd.read_csv(text_dataset_path, encoding='utf-8')
    except UnicodeDecodeError:
        print("  UTF-8 failed, trying latin-1...")
        df_text = pd.read_csv(text_dataset_path, encoding='latin-1')
    df_text = df_text.head(num_records)
    print(f"✓ Text dataset loaded: {len(df_text):,} records")
    
    print(f" Loading URL dataset from: {url_dataset_path}")
    try:
        df_url = pd.read_csv(url_dataset_path, encoding='utf-8')
    except UnicodeDecodeError:
        print("   UTF-8 failed, trying latin-1...")
        df_url = pd.read_csv(url_dataset_path, encoding='latin-1')
    df_url = df_url.head(num_records)
    print(f"✓ URL dataset loaded: {len(df_url):,} records")
    
    # Extract data
    texts = df_text[text_col].fillna("").astype(str).tolist()
    urls_list = df_url[url_col].fillna("").astype(str).tolist()
    
    # Combine: text samples + URL samples
    all_texts = texts + [""] * len(urls_list)
    all_urls = [""] * len(texts) + urls_list
    
    print(f"\n Processing {len(all_texts):,} total records")
    print("-" * 70)
    print(" Processing with batch optimization...")
    
    start_time = time.perf_counter()
    results = predict_batch(all_texts, all_urls, batch_size=batch_size, show_progress=True)
    total_time = time.perf_counter() - start_time
    
    # Results summary
    print("\n" + "=" * 70)
    print(" RESULTS SUMMARY")
    print("=" * 70)
    
    print(f"\n VERALL RUNTIME:")
    print(f"  Total time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
    print(f"  Average per record: {(total_time/len(all_texts))*1000:.2f} ms")
    print(f"  Throughput: {len(all_texts)/total_time:.2f} records/second")
    
    print(f"\n COMPONENT TIMING (Average per record):")
    print(f"  Preprocessing:    {np.mean(results['timings']['preprocessing'])*1000:.2f} ms")
    print(f"  URL Expert:       {np.mean(results['timings']['url_expert'])*1000:.2f} ms")
    print(f"  Text Expert:      {np.mean(results['timings']['text_expert'])*1000:.2f} ms")
    print(f"  Gating Network:   {np.mean(results['timings']['gating_network'])*1000:.2f} ms")
    print(f"  Combination:      {np.mean(results['timings']['combination'])*1000:.2f} ms")
    
    # Prediction distribution
    phishing_count = sum(1 for p in results['predictions'] if 'PHISHING' in p)
    
    print(f"\n PREDICTION DISTRIBUTION:")
    print(f"  PHISHING: {phishing_count:,} ({phishing_count/len(all_texts)*100:.1f}%)")
    print(f"  SAFE:     {len(all_texts)-phishing_count:,} ({(len(all_texts)-phishing_count)/len(all_texts)*100:.1f}%)")
    
    # Breakdown by type
    text_phishing = sum(1 for i in range(len(texts)) if 'PHISHING' in results['predictions'][i])
    url_phishing = sum(1 for i in range(len(texts), len(all_texts)) if 'PHISHING' in results['predictions'][i])
    
    print(f"\n BREAKDOWN BY TYPE:")
    print(f"  Text samples: {len(texts):,} | PHISHING: {text_phishing:,} ({text_phishing/len(texts)*100:.1f}%)")
    print(f"  URL samples:  {len(urls_list):,} | PHISHING: {url_phishing:,} ({url_phishing/len(urls_list)*100:.1f}%)")
    
    print(f"\n CONFIDENCE STATISTICS:")
    print(f"  Average: {np.mean(results['confidences']):.2f}%")
    print(f"  Min: {np.min(results['confidences']):.2f}%")
    print(f"  Max: {np.max(results['confidences']):.2f}%")
    
    print("=" * 70)
    
    return results



Loading Expert Models...
✓ Expert 1 (URL) loaded
✓ Expert 2 (Text) loaded on cuda
Loading Trained Gating Network...
✓ Gating Network loaded

✅ Complete MoE system loaded!


=======================OPTIMIZED SYSTEM READY! =======================

 Available Commands:
----------------------------------------------------------------------
1. Single prediction:
   test_sample("URGENT! Click here http://paypa1.com")

2. 🚀 OPTIMIZED batch processing (RECOMMENDED):
   results = test_large_dataset(
       text_dataset_path=r"C:\...\Dataset_100k.csv",
       url_dataset_path=r"C:\...\phishing_site_urls.csv",
       num_records=10000,
       text_col="TEXT",
       url_col="URL",
       batch_size=64  # Try 32, 64, 128, or 256
   )


In [8]:
# Run the optimized test
results = test_large_dataset(
    text_dataset_path=r"C:\Users\angelo\Downloads\THESIS\Dataset_100k.csv",
    url_dataset_path=r"C:\Users\angelo\Downloads\THESIS\phishing_site_urls.csv",
    num_records=5000,
    text_col="text",
    url_col="URL",
    batch_size=64  # Try 32, 64, 128, or 256
)


              OPTIMIZED BATCH PROCESSING (batch_size=64)              
                             Device: cuda                             
  Loading text dataset from: C:\Users\angelo\Downloads\THESIS\Dataset_100k.csv
  UTF-8 failed, trying latin-1...
✓ Text dataset loaded: 5,000 records
 Loading URL dataset from: C:\Users\angelo\Downloads\THESIS\phishing_site_urls.csv
✓ URL dataset loaded: 5,000 records

 Processing 10,000 total records
----------------------------------------------------------------------
 Processing with batch optimization...
  Processed 8,000/10,000 | Rate: 314.1 rec/sec
  Processed 10,000/10,000 | Rate: 389.9 rec/sec

 RESULTS SUMMARY

 VERALL RUNTIME:
  Total time: 25.64 seconds (0.43 minutes)
  Average per record: 2.56 ms
  Throughput: 389.95 records/second

 COMPONENT TIMING (Average per record):
  Preprocessing:    0.01 ms
  URL Expert:       0.03 ms
  Text Expert:      2.49 ms
  Gating Network:   0.03 ms
  Combination:      0.01 ms

 PREDICTION DISTRIBUTI